# 04 — Figures (layout preview)

Figure scaffolding for the Dual-Duty Tax analysis, with an explicit **`USE_FAKE`** switch so Isaac can develop against a synthetic fixture before Sid's `route_features.parquet` handoff arrives.

- **Fake mode** reads `data/route_features_FAKE.parquet` and writes previews only to `figures/mock/` with a SYNTHETIC watermark.
- **Real mode** reads `data/route_features.parquet` and writes to `figures/`. It **fails clearly** if the handoff is missing and **never falls back** to synthetic data.

In [ ]:
import sys
from pathlib import Path

for _p in [Path.cwd(), *Path.cwd().parents]:
    if (_p / 'src' / 'config.py').exists():
        sys.path.insert(0, str(_p / 'src'))
        _ROOT = _p
        break
else:
    raise SystemExit('Could not locate src/config.py from cwd.')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import (
    DATA_DIR, ROOT, TE_RB_POSITIONS, RB_POSITIONS,
    END_EVENTS, NO_THROW_EVENTS,
)
print('project root:', ROOT)


## Mode switch

In [ ]:
# === MODE SWITCH ============================================================
USE_FAKE = True   # True  -> synthetic fixture, figures/mock/ only
                  # False -> real handoff data/route_features.parquet, figures/
# ============================================================================

REQUIRED_COLS = ['gameId','playId','nflId','group','position',
                 't_throw','t_release','depth_at_throw','sep_at_throw','end_event']
print('USE_FAKE =', USE_FAKE)


## Input checks (no silent fallback)

In [ ]:
if USE_FAKE:
    DATA_PATH = DATA_DIR / 'route_features_FAKE.parquet'
    FIG_DIR = ROOT / 'figures' / 'mock'
    WATERMARK = 'SYNTHETIC DEMO - NOT RESULTS'
else:
    DATA_PATH = DATA_DIR / 'route_features.parquet'
    FIG_DIR = ROOT / 'figures'
    WATERMARK = None

if not DATA_PATH.exists():
    if USE_FAKE:
        raise FileNotFoundError(
            f'Synthetic fixture missing: {DATA_PATH}. '
            f'Run notebooks/make_fake_fixture.py to generate it.')
    else:
        raise FileNotFoundError(
            f'Real handoff missing: {DATA_PATH}. '
            f'Sid has not delivered data/route_features.parquet yet. '
            f'Set USE_FAKE=True to develop against the synthetic fixture instead. '
            f'There is no automatic fallback to synthetic data in real mode.')
print('data path:', DATA_PATH)
print('figure dir:', FIG_DIR)


## Load & validate schema

In [ ]:
df = pd.read_parquet(DATA_PATH)

# schema + dtype checks
assert list(df.columns) == REQUIRED_COLS, f'columns mismatch: {list(df.columns)}'
for c in ['gameId','playId','nflId']:
    assert pd.api.types.is_integer_dtype(df[c]), f'{c} not integer'
for c in ['t_throw','t_release','depth_at_throw','sep_at_throw']:
    assert pd.api.types.is_float_dtype(df[c]), f'{c} not float'
assert set(df['group'].dropna().unique()).issubset({'chip','leak','clean'})
assert set(df['position'].dropna().unique()).issubset(set(TE_RB_POSITIONS))
assert set(df['end_event'].dropna().unique()).issubset(set(END_EVENTS))

# key uniqueness
assert df.duplicated(['gameId','playId','nflId']).sum() == 0, 'duplicate keys'

# timing / value constraints
assert (df['t_throw'] >= 0).all(), 'negative t_throw'
assert (df['sep_at_throw'] >= 0).all(), 'negative sep_at_throw'
_obs = df['t_release'].notna()
assert (df.loc[_obs,'t_release'] >= 0).all(), 'negative observed t_release'
assert (df.loc[_obs,'t_release'] <= df.loc[_obs,'t_throw']).all(), 't_release > t_throw'

print('rows:', len(df))
print('validation OK')
print('group counts:'); print(df['group'].value_counts().to_string())
print('end_event counts:'); print(df['end_event'].value_counts().to_string())
print('missing t_release:', int(df['t_release'].isna().sum()))


## Layout preview (NOT findings)

In [ ]:
FIG_DIR.mkdir(parents=True, exist_ok=True)
fig, axes = plt.subplots(1, 3, figsize=(11,3.2))

# panel 1: group counts
df['group'].value_counts().reindex(['chip','leak','clean']).plot.bar(ax=axes[0], color='#555')
axes[0].set_title('route group counts')
axes[0].set_xlabel('group')

# panel 2: time to endpoint by group
df.boxplot(column='t_throw', by='group', ax=axes[1], grid=False)
axes[1].set_title('time to endpoint by group')
axes[1].set_xlabel('group'); axes[1].set_ylabel('t_throw (s)')
plt.suptitle('')

# panel 3: separation distribution
axes[2].hist(df['sep_at_throw'], bins=20, color='#888')
axes[2].set_title('separation at throw')
axes[2].set_xlabel('sep_at_throw (yd)')

if WATERMARK:
    fig.text(0.5, 0.5, WATERMARK, ha='center', va='center',
             fontsize=22, color='red', alpha=0.18, rotation=30)
    fig.suptitle('SYNTHETIC DEMO - NOT RESULTS', color='red')

plt.tight_layout()
out = FIG_DIR / ('preview_groups' + ('_FAKE' if USE_FAKE else '') + '.png')
plt.savefig(out, dpi=120)
plt.show()
print('saved:', out)


## Interpretation

Do not read the panels above as findings. They are a **layout preview** to confirm the figure plumbing works on the schema. Real analysis and final figures are produced only after Sid's `route_features.parquet` handoff lands, with `USE_FAKE=False`.

> Synthetic previews live only in `figures/mock/`. Real mode writes to `figures/` and fails loudly if the handoff is missing.